# MPRF — Five seeds, α/k sensitivity and ablations
Place the four data files in `/content` (same names as in the other notebooks), then run the cell below. Outputs: raw CSV files.
Seed 0 reproduces the protocol of the original notebook; seeds 0–4 give the five-seed means reported in the paper.

In [ ]:
# ============================================================
# MPRF — Expériences complémentaires (R1.4, R1.5, R1.6, R1.7, R1.8)
#  1. Sensibilité à alpha (0 = G-mean seul ... 1 = F1 seul) et au budget k
#  2. Ablations : sans class_weight, F1 seul, G-mean seul, init "meilleur arbre"
#  3. Plusieurs seeds (seed 0 = protocole exact du papier)
#  4. alpha choisi par validation imbriquée (nested) sur le train de chaque fold
#  5. Sensibilité, spécificité, FP, FN, intervalles de confiance (bootstrap)
#  6. Tests de Wilcoxon avec correction de Holm + différences appariées
# Aucun fichier du papier n'est modifié. Google Colab — copier-coller directement
# ============================================================

import os, time
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import average_precision_score
from sklearn.preprocessing import LabelEncoder
from scipy.stats import wilcoxon
import warnings
warnings.filterwarnings("ignore")

# ┌──────────────────────── PARAMÈTRES ────────────────────────┐
DATASETS = ["cervical", "hepatitis", "pima", "mammography"]   # retirer ceux que tu ne veux pas relancer
SEEDS    = [0, 1, 2, 3, 4]          # seed 0 = exactement le protocole du papier
ALPHAS   = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
KS       = [5, 10, 15, 20]
ALPHA0, K0 = 0.6, 10                # configuration du papier
N_TREES, N_SPLITS = 50, 10
NESTED, NESTED_SEEDS, INNER_SPLITS = True, [0], 3   # alpha choisi par CV interne (3 plis) sur le train
N_BOOT   = 2000
OUT_DIR  = "."                      # les CSV sont écrits ici
FILES = {                           # mêmes fichiers que dans ton notebook
    "cervical":    "1. risk_factors_cervical_cancer.csv",
    "hepatitis":   "2. HepatitisCdata.csv",
    "pima":        "3. diabetes.csv",
    "mammography": "/content/phpn1jVwe.arff",
}
# └─────────────────────────────────────────────────────────────┘

# ── Chargement (identique à ton notebook) ───────────────────
def load(name):
    path = FILES[name]
    if not os.path.exists(path):
        print(f"⚠️  {name}: fichier introuvable ({path}) → ignoré"); return None
    if name == "cervical":
        df = pd.read_csv(path); df.replace("?", np.nan, inplace=True)
        df = df.apply(pd.to_numeric, errors="coerce"); df.fillna(df.median(numeric_only=True), inplace=True)
        feats = [c for c in df.columns if c != "Biopsy"]
        return df[feats].values, df["Biopsy"].values.astype(int)
    if name == "hepatitis":
        df = pd.read_csv(path); df["Category"] = df["Category"].astype(str).str.strip()
        df["Category"] = df["Category"].map(lambda x: 0 if "Blood Donor" in x else (1 if "Hepatitis" in x else np.nan))
        df = df.dropna(subset=["Category"])
        feats = [c for c in df.columns if c not in ["Category", "Unnamed: 0"]]
        d = df[feats].copy()
        for col in d.columns:
            if d[col].dtype == object:
                le = LabelEncoder(); m = d[col].notna()
                d.loc[m, col] = le.fit_transform(d.loc[m, col].astype(str)); d[col] = pd.to_numeric(d[col], errors="coerce")
        d = d.apply(pd.to_numeric, errors="coerce"); d.fillna(d.median(numeric_only=True), inplace=True)
        return d.values, df["Category"].values.astype(int)
    if name == "pima":
        df = pd.read_csv(path); df.fillna(df.median(numeric_only=True), inplace=True)
        return df[[c for c in df.columns if c != "Outcome"]].values, df["Outcome"].values.astype(int)
    if name == "mammography":
        rows, data = [], False
        for line in open(path):
            line = line.strip()
            if line.lower() == "@data": data = True; continue
            if data and line:
                v = line.split(","); rows.append([float(t) for t in v[:6]] + [1 if v[6].strip().strip("'") == "1" else 0])
        a = np.array(rows); return a[:, :6], a[:, 6].astype(int)

# ── Métriques / score (mêmes formules que le notebook) ──────
def counts(y, p):
    tp = int(((y == 1) & (p == 1)).sum()); fp = int(((y == 0) & (p == 1)).sum())
    fn = int(((y == 1) & (p == 0)).sum()); tn = int(((y == 0) & (p == 0)).sum())
    return tp, fp, fn, tn

def f1_gm(tp, fp, fn, tn):
    den = 2 * tp + fp + fn
    f1 = 2 * tp / den if den > 0 else 0.0
    r1 = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    r0 = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    return f1, np.sqrt(r1 * r0)

def metrics(y, p, proba):
    tp, fp, fn, tn = counts(y, p); f1, gm = f1_gm(tp, fp, fn, tn)
    return dict(f1_min=f1, gmean=gm,
                precision=tp / (tp + fp) if (tp + fp) > 0 else 0.0,
                sens=tp / (tp + fn) if (tp + fn) > 0 else 0.0,
                spec=tn / (tn + fp) if (tn + fp) > 0 else 0.0,
                fp=fp, fn=fn, auprc=average_precision_score(y, proba))

def score_vec(yv, pred_mat, alpha):          # pred_mat : (candidats, n_val) booléens
    pos = (yv == 1); neg = ~pos
    tp = (pred_mat & pos).sum(1); fp = (pred_mat & neg).sum(1)
    fn = (~pred_mat & pos).sum(1); tn = (~pred_mat & neg).sum(1)
    den = 2 * tp + fp + fn
    f1 = np.where(den > 0, 2 * tp / np.maximum(den, 1), 0.0)
    r1 = np.where(tp + fn > 0, tp / np.maximum(tp + fn, 1), 0.0)
    r0 = np.where(tn + fp > 0, tn / np.maximum(tn + fp, 1), 0.0)
    return alpha * f1 + (1 - alpha) * np.sqrt(r1 * r0)

# ── Sélection d'arbres (P = matrice (arbres, instances) des prédictions 0/1) ──
def MPRF(Pv, yv, k, alpha, init_seed, init="random"):
    n = Pv.shape[0]
    if init == "random":
        i0 = np.random.RandomState(init_seed).randint(0, n)
    else:                                         # "best" : meilleur arbre seul sur la validation
        i0 = int(np.argmax(score_vec(yv, Pv.astype(bool), alpha)))
    sel = [i0]; rem = [i for i in range(n) if i != i0]
    votes = Pv[i0].astype(float).copy()
    best = score_vec(yv, (votes >= 0.5)[None, :], alpha)[0]
    while len(sel) < k and rem:
        V = votes[None, :] + Pv[rem]
        sc = score_vec(yv, V >= (len(sel) + 1) / 2, alpha)
        j = int(np.argmax(sc))                    # 1er maximum = même règle que la boucle d'origine
        if sc[j] >= best:                         # plateau accepté
            votes = V[j]; sel.append(rem.pop(j))
            if sc[j] > best: best = sc[j]
        else:
            break
    return sel

def OBP(Pv, yv, max_size):
    n = Pv.shape[0]
    sc = [f1_gm(*counts(yv, Pv[i]))[0] for i in range(n)]
    order = np.argsort(sc)[::-1]; sel, best, votes = [], -np.inf, np.zeros(Pv.shape[1])
    for idx in order[:max_size]:
        v = votes + Pv[idx]; s = f1_gm(*counts(yv, (v >= (len(sel) + 1) / 2).astype(int)))[0]
        if s >= best: sel.append(idx); votes = v; best = s
    return sel if sel else [order[0]]

def IBEP(Pv, yv, max_size):
    n = Pv.shape[0]; err = (Pv != yv).astype(float); w = np.ones(len(yv)) / len(yv); sel = []
    for _ in range(max_size):
        b = int(np.argmin(err.dot(w)))
        if b in sel: break
        sel.append(b); w *= np.exp(err[b]); w /= w.sum()
    return sel if sel else [0]

def vote(P, sel):  return (P[sel].sum(0) >= len(sel) / 2).astype(int)

def forest(Xa, ya, seed, balanced):
    return RandomForestClassifier(n_estimators=N_TREES, class_weight="balanced" if balanced else None,
                                  random_state=42 + seed, n_jobs=-1).fit(Xa, ya)

def tree_mats(rf, Xv, Xt):
    Pv = np.array([t.predict(Xv) for t in rf.estimators_])
    Pt = np.array([t.predict(Xt) for t in rf.estimators_])
    Qt = np.array([t.predict_proba(Xt)[:, 1] for t in rf.estimators_])
    return Pv, Pt, Qt

# ── alpha par validation imbriquée (n'utilise que le train du pli externe) ──
def nested_alpha(Xtr, ytr, fold, seed):
    skf = StratifiedKFold(INNER_SPLITS, shuffle=True, random_state=seed + 7)
    acc = {a: [] for a in ALPHAS}
    for j, (itr, ite) in enumerate(skf.split(Xtr, ytr)):
        rng = np.random.RandomState(fold * 31 + j + 1000 * seed)
        vi = rng.choice(len(itr), int(len(itr) * 0.2), replace=False); mask = np.ones(len(itr), bool); mask[vi] = False
        rf = forest(Xtr[itr][mask], ytr[itr][mask], seed, True)
        Pv, Pt, _ = tree_mats(rf, Xtr[itr][vi], Xtr[ite])
        for a in ALPHAS:
            sel = MPRF(Pv, ytr[itr][vi], K0, a, 100 + fold + 1000 * seed)
            acc[a].append(f1_gm(*counts(ytr[ite], vote(Pt, sel)))[0])
    m = {a: np.mean(v) for a, v in acc.items()}
    return max(ALPHAS, key=lambda a: (m[a], -abs(a - ALPHA0)))

# ── Boucle principale ───────────────────────────────────────
def run(name, X, y):
    rows = []; skf = StratifiedKFold(N_SPLITS, shuffle=True, random_state=42)
    for seed in SEEDS:
        for fold, (tr, te) in enumerate(skf.split(X, y)):
            Xtr, Xte, ytr, yte = X[tr], X[te], y[tr], y[te]
            vi = np.random.RandomState(fold + 1000 * seed).choice(len(ytr), int(len(tr) * 0.2), replace=False)
            sub = np.setdiff1d(np.arange(len(ytr)), vi)
            Xv, yv, Xs, ys = Xtr[vi], ytr[vi], Xtr[sub], ytr[sub]
            iseed = 100 + fold + 1000 * seed
            def add(method, pred, proba, size):
                d = metrics(yte, pred, proba); d.update(dataset=name, seed=seed, fold=fold, method=method, size=size); rows.append(d)
            for tag, bal in (("cs", True), ("nocw", False)):
                rf = forest(Xs, ys, seed, bal); Pv, Pt, Qt = tree_mats(rf, Xv, Xte)
                if bal:
                    add("CSRF", rf.predict(Xte), rf.predict_proba(Xte)[:, 1], N_TREES)
                    for nm, sel in (("OBP", OBP(Pv, yv, N_TREES)), ("IBEP", IBEP(Pv, yv, N_TREES))):
                        add(nm, vote(Pt, sel), Qt[sel].mean(0), len(sel))
                    for a in ALPHAS:
                        sel = MPRF(Pv, yv, K0, a, iseed); add(f"MPRF a={a:.1f} k={K0}", vote(Pt, sel), Qt[sel].mean(0), len(sel))
                    for k in KS:
                        if k != K0:
                            sel = MPRF(Pv, yv, k, ALPHA0, iseed); add(f"MPRF a={ALPHA0:.1f} k={k}", vote(Pt, sel), Qt[sel].mean(0), len(sel))
                    sel = MPRF(Pv, yv, K0, ALPHA0, iseed, init="best"); add("MPRF init=best-tree", vote(Pt, sel), Qt[sel].mean(0), len(sel))
                    if NESTED and seed in NESTED_SEEDS:
                        a_star = nested_alpha(Xtr, ytr, fold, seed)
                        sel = MPRF(Pv, yv, K0, a_star, iseed); add("MPRF nested-alpha", vote(Pt, sel), Qt[sel].mean(0), len(sel))
                        rows[-1]["alpha_star"] = a_star
                else:
                    add("RF no class_weight", rf.predict(Xte), rf.predict_proba(Xte)[:, 1], N_TREES)
                    sel = MPRF(Pv, yv, K0, ALPHA0, iseed); add("MPRF on no-class_weight forest", vote(Pt, sel), Qt[sel].mean(0), len(sel))
        print(f"   {name}: seed {seed} terminé", flush=True)
    return pd.DataFrame(rows)

# ── Statistiques ────────────────────────────────────────────
rng_b = np.random.RandomState(0)
def boot_ci(v, n=N_BOOT):
    v = np.asarray(v, float); m = [rng_b.choice(v, len(v)).mean() for _ in range(n)]
    return np.percentile(m, 2.5), np.percentile(m, 97.5)

def cliff(x, y):
    x, y = np.asarray(x), np.asarray(y); return (sum(a > b for a in x for b in y) - sum(a < b for a in x for b in y)) / (len(x) * len(y))

def holm(ps):
    ps = np.array(ps, float); o = np.argsort(ps); adj = np.empty_like(ps); run_max = 0.0; m = len(ps)
    for r, i in enumerate(o):
        run_max = max(run_max, min(1.0, (m - r) * ps[i])); adj[i] = run_max
    return adj

MAIN = f"MPRF a={ALPHA0:.1f} k={K0}"
def report(R, name):
    D = R[R.dataset == name]
    print("\n" + "=" * 100); print(f"  {name.upper()}   (moyenne sur les plis ; ± = écart-type entre seeds ; [ ] = IC95% bootstrap sur les 10 plis)"); print("=" * 100)
    per = D.groupby(["method", "seed"]).mean(numeric_only=True).reset_index()
    foldavg = D.groupby(["method", "fold"]).mean(numeric_only=True).reset_index()   # moyenne sur seeds, par pli
    def line(m):
        p = per[per.method == m]; f = foldavg[foldavg.method == m]
        s = lambda c: f"{p[c].mean():.3f}±{p[c].std(ddof=0):.3f}"
        lo, hi = boot_ci(f["f1_min"]); return (f"{m:<32} n={p['size'].mean():4.1f} | F1={s('f1_min')} [{lo:.3f},{hi:.3f}] | Gm={s('gmean')} | "
               f"AUPRC={s('auprc')} | Sens={p['sens'].mean():.3f} Spec={p['spec'].mean():.3f} Prec={p['precision'].mean():.3f} | FP={p['fp'].mean():5.1f} FN={p['fn'].mean():4.1f}")
    groups = [("A. Méthodes principales (R1.7, R1.8)", ["CSRF", "OBP", "IBEP", MAIN]),
              ("B. Sensibilité à alpha, k=10 (R1.4, R1.7) — alpha=0: G-mean seul, alpha=1: F1 seul", [f"MPRF a={a:.1f} k={K0}" for a in ALPHAS]),
              ("C. Budget d'arbres k, alpha=0.6 (R1.7)", [f"MPRF a={ALPHA0:.1f} k={k}" for k in KS]),
              ("D. Entraînement sensible au coût (R1.7)", ["RF no class_weight", "MPRF on no-class_weight forest", "CSRF", MAIN]),
              ("E. Initialisation (R1.6) — aléatoire (variabilité entre seeds) vs meilleur arbre", [MAIN, "MPRF init=best-tree"])]
    if (D.method == "MPRF nested-alpha").any():
        groups.append(("F. alpha choisi par validation imbriquée (seed 0) (R1.4)", ["MPRF nested-alpha"]))
    for title, ms in groups:
        print(f"\n  {title}")
        for m in ms:
            if (per.method == m).any(): print("   " + line(m))
    if (D.method == "MPRF nested-alpha").any():
        a = D[D.method == "MPRF nested-alpha"].sort_values("fold")["alpha_star"].astype(float).values
        s0 = D[(D.method == MAIN) & (D.seed == 0)].sort_values("fold"); n0 = D[(D.method == "MPRF nested-alpha") & (D.seed == 0)].sort_values("fold")
        print(f"   alpha choisi par pli : {a.tolist()}")
        print(f"   seed 0 — F1 alpha=0.6 fixe : {s0.f1_min.mean():.3f} | F1 alpha imbriqué : {n0.f1_min.mean():.3f} | G-mean : {s0.gmean.mean():.3f} vs {n0.gmean.mean():.3f}")
    # tests appariés
    print("\n  G. MPRF vs baselines — Wilcoxon sur valeurs par pli (moyenne des seeds), p brut et p Holm (9 tests) (R1.8)")
    out = []
    for met in ["f1_min", "gmean", "auprc"]:
        x = foldavg[foldavg.method == MAIN].sort_values("fold")[met].values
        for b in ["CSRF", "OBP", "IBEP"]:
            y_ = foldavg[foldavg.method == b].sort_values("fold")[met].values
            try: p = 1.0 if np.all(x == y_) else wilcoxon(x, y_)[1]
            except Exception: p = np.nan
            d = x - y_; dm = [rng_b.choice(d, len(d)).mean() for _ in range(N_BOOT)]
            out.append((met, b, d.mean(), np.percentile(dm, 2.5), np.percentile(dm, 97.5), p, cliff(x, y_)))
    adj = holm([o[5] for o in out])
    print(f"   {'métrique':<8} {'vs':<6} {'Δ moyen':>8} {'IC95% Δ':>18} {'p brut':>8} {'p Holm':>8} {'Cliff δ':>8}")
    for o, a_ in zip(out, adj):
        print(f"   {o[0]:<8} {o[1]:<6} {o[2]:>8.3f} [{o[3]:>7.3f},{o[4]:>7.3f}] {o[5]:>8.4f} {a_:>8.4f} {o[6]:>8.3f}")

t0 = time.time(); allres = []
for name in DATASETS:
    ld = load(name)
    if ld is None: continue
    X, y = ld; c = np.bincount(y)
    print(f"\n### {name} : {X.shape} | classes {c.tolist()} | IR={c[0]/c[1]:.1f}:1 | minoritaires par pli de test ≈ {c[1]/N_SPLITS:.1f}")
    R = run(name, X, y); allres.append(R); report(pd.concat(allres), name)
    pd.concat(allres).to_csv(os.path.join(OUT_DIR, "mprf_extra_experiments_raw.csv"), index=False)
print(f"\nTerminé en {(time.time()-t0)/60:.1f} min — résultats bruts : mprf_extra_experiments_raw.csv")
print("Vérification : la ligne 'CSRF' / 'MPRF a=0.6 k=10' doit retrouver les valeurs du papier quand seed 0 est seul (SEEDS=[0]).")
